# 3. Historical Census Sectors Evolution Analysis (2010-2022)

This section analyzes the evolution of census sectors in Belo Horizonte using IBGE historical formation data. We'll examine how sectors changed between censuses, which is crucial for understanding demographic continuity in epidemiological research.

**Dataset:** `historico_setores_censitarios_2010_2022.csv`
**Scope:** Nationwide evolution tracking with focus on Belo Horizonte
**Period:** 2010 → 2022 census transformation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Define global color palette for visualizations
colors = ['#2E8B57', '#FF6B35', '#F7931E', '#FFD23F', '#06FFA5', '#4ECDC4', '#45B7D1', '#96CEB4', '#FF9999', '#DDA0DD', '#98FB98', '#F0E68C', '#FFB6C1', '#87CEEB', '#DEB887']

print("Color palette defined successfully!")

## 3.0 Load Required Data Dependencies

This notebook requires data from other IBGE analysis notebooks. We'll load the necessary datasets or provide fallback options.

In [ ]:
# Load required datasets from raw IBGE data (same approach as IBGE_data_exploration.ipynb)
print("🔄 Loading Required Data Dependencies from Raw IBGE Data")
print("=" * 60)

try:
    import geopandas as gpd
    
    # === LOAD 2010 DATA ===
    print("\n📅 Loading 2010 Census Data...")
    
    # 2010 population data
    data_2010_path = Path("../../data/IBGE/2010/population/Base_informacoes_setores2010_sinopse_MG.xls")
    if data_2010_path.exists():
        print(f"✅ Loading 2010 population data: {data_2010_path.name}")
        df_2010 = pd.read_excel(data_2010_path, sheet_name=0)
        
        # Filter for Belo Horizonte
        belo_horizonte_2010 = df_2010[df_2010['Nome_do_municipio'].str.contains('Belo Horizonte', case=False, na=False)].copy()
        print(f"   • 2010 BH sectors: {len(belo_horizonte_2010):,}")
    else:
        print("⚠️ 2010 population data not found")
        belo_horizonte_2010 = None
    
    # 2010 shapefile data
    shapefile_2010_dir = Path("../../data/IBGE/2010/shapefiles")
    if shapefile_2010_dir.exists():
        print(f"✅ Loading 2010 shapefiles from: {shapefile_2010_dir}")
        
        # Get BH subdistrito codes from 2010 data
        if belo_horizonte_2010 is not None:
            bh_subdistrito_codes = [31062000562, 31062000563, 31062000564, 31062000565, 31062000567, 
                                   31062000568, 31062000569, 31062002561, 31062002567, 31062006064, 
                                   31062006066, 31062006068, 31062006069]
            
            setor_gdfs_2010 = []
            for code in bh_subdistrito_codes:
                setor_path = shapefile_2010_dir / str(code) / f"{code}_setor.shp"
                if setor_path.exists():
                    gdf_setor = gpd.read_file(setor_path)
                    gdf_setor['Cod_subdistrito'] = code
                    setor_gdfs_2010.append(gdf_setor)
            
            if setor_gdfs_2010:
                bh_setores_gdf = pd.concat(setor_gdfs_2010, ignore_index=True).to_crs(epsg=4326)
                print(f"   • 2010 BH sectors shapefile: {len(bh_setores_gdf):,} sectors")
            else:
                print("⚠️ No 2010 sector shapefiles found")
                bh_setores_gdf = None
        else:
            bh_setores_gdf = None
    else:
        print("⚠️ 2010 shapefiles directory not found")
        bh_setores_gdf = None
    
    # === LOAD 2022 DATA ===
    print("\n📅 Loading 2022 Census Data...")
    
    # 2022 population data
    data_2022_path = Path("../../data/IBGE/2022/population/Agregados_por_setores_basico_BR_20250417.csv")
    if data_2022_path.exists():
        print(f"✅ Loading 2022 population data: {data_2022_path.name}")
        df_2022 = pd.read_csv(data_2022_path, encoding='latin-1', sep=';')
        
        # Filter for Belo Horizonte
        bh_mask_2022 = df_2022['CD_SETOR'].astype(str).str.startswith('3106200')
        belo_horizonte_2022 = df_2022[bh_mask_2022].copy()
        print(f"   • 2022 BH sectors: {len(belo_horizonte_2022):,}")
    else:
        print("⚠️ 2022 population data not found")
        belo_horizonte_2022 = None
    
    # 2022 shapefile data
    shapefile_2022_dir = Path("../../data/IBGE/2022/shapefiles")
    mg_shapefile_2022 = shapefile_2022_dir / "MG_setores_CD2022.shp"
    
    if mg_shapefile_2022.exists():
        print(f"✅ Loading 2022 shapefile: {mg_shapefile_2022.name}")
        mg_sectors_gdf_2022 = gpd.read_file(mg_shapefile_2022)
        
        # Filter for Belo Horizonte
        if 'CD_SETOR' in mg_sectors_gdf_2022.columns:
            bh_mask_shp = mg_sectors_gdf_2022['CD_SETOR'].astype(str).str.startswith('3106200')
        elif 'CD_GEOCODI' in mg_sectors_gdf_2022.columns:
            bh_mask_shp = mg_sectors_gdf_2022['CD_GEOCODI'].astype(str).str.startswith('3106200')
        else:
            # Find sector column
            sector_cols = [col for col in mg_sectors_gdf_2022.columns if 'CD_' in col]
            if sector_cols:
                sector_col = sector_cols[0]
                bh_mask_shp = mg_sectors_gdf_2022[sector_col].astype(str).str.startswith('3106200')
            else:
                print("⚠️ No sector code column found in 2022 shapefile")
                bh_mask_shp = pd.Series([False] * len(mg_sectors_gdf_2022))
        
        sectors_with_data_2022 = mg_sectors_gdf_2022[bh_mask_shp].copy().to_crs(epsg=4326)
        print(f"   • 2022 BH sectors shapefile: {len(sectors_with_data_2022):,} sectors")
    else:
        print("⚠️ 2022 shapefile not found")
        sectors_with_data_2022 = None
    
    print(f"\n📊 Final Data Status:")
    print(f"   • 2010 population data: {'Available' if belo_horizonte_2010 is not None else 'Missing'}")
    print(f"   • 2010 shapefile data: {'Available' if bh_setores_gdf is not None else 'Missing'}")
    print(f"   • 2022 population data: {'Available' if belo_horizonte_2022 is not None else 'Missing'}")
    print(f"   • 2022 shapefile data: {'Available' if sectors_with_data_2022 is not None else 'Missing'}")
    
except ImportError:
    print("⚠️ GeoPandas not available - geographic visualizations will be limited")
    bh_setores_gdf = None
    sectors_with_data_2022 = None
    belo_horizonte_2010 = None
    belo_horizonte_2022 = None
except Exception as e:
    print(f"⚠️ Error loading census data: {e}")
    bh_setores_gdf = None
    sectors_with_data_2022 = None
    belo_horizonte_2010 = None
    belo_horizonte_2022 = None


## 3.1 Load Historical Formation Data


In [ ]:
# Load historical census sectors evolution data
print("📂 Loading Historical Census Sectors Evolution Data")
print("=" * 60)

# Check if CSV exists - corrected path
csv_path = Path("../../data/IBGE/historico_setores_censitarios_2010_2022.csv")
print(f"File path: {csv_path}")
print(f"File exists: {csv_path.exists()}")

if csv_path.exists():
    # Load the data
    df_evolution = pd.read_csv(csv_path)
    
    print(f"\n✅ Data loaded successfully!")
    print(f"   📊 Shape: {df_evolution.shape[0]:,} rows × {df_evolution.shape[1]} columns")
    print(f"   💾 Memory usage: {df_evolution.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Display column information
    print(f"\n📋 Dataset structure:")
    print(f"   • Tracks sector evolution from 2010 to 2022")
    print(f"   • Contains {len(df_evolution.columns)} temporal snapshots")
    print(f"   • Covers entire Brazil ({len(df_evolution):,} records)")
    
    # Show key columns
    key_columns = ['GEOCODIGO_2022_DIVULGAÇÃO', 'FRM_2022_DIVULGAÇÃO', 'GEOCODIGO_2010']
    description = ["Code in 2022", "Formation type in 2022", "Code in 2010"]
    print(f"\n🔑 Key columns for analysis:")
    for col in key_columns:
        if col in df_evolution.columns:
            print(f"   • {col}: Sector codes and formation types")
    
    # Display first few rows
    print(f"\n📊 Data preview:")
    display(df_evolution[key_columns].head())
    
else:
    print(f"❌ File not found!")
    print(f"   Expected location: {csv_path}")
    print(f"   Please ensure the Excel file has been converted to CSV format.")
    df_evolution = None


## 3.2 Belo Horizonte Sector Evolution Analysis



In [ ]:
if df_evolution is not None:
    print("🏙️ Belo Horizonte Sectors Evolution Analysis")
    print("=" * 55)
    
    # Filter for Belo Horizonte (municipal code: 3106200)
    # 2022 sectors
    bh_2022_hist = df_evolution[df_evolution['GEOCODIGO_2022_DIVULGAÇÃO'].astype(str).str.startswith('3106200')]
    # 2010 sectors 
    bh_2010_hist = df_evolution[df_evolution['GEOCODIGO_2010'].astype(str).str.startswith('3106200')]
    
    print(f"📈 Sector count evolution:")
    print(f"   • 2010 Census: {len(bh_2010_hist):,} sectors")
    print(f"   • 2022 Census: {len(bh_2022_hist):,} sectors")
    
    sector_change = len(bh_2022_hist) - len(bh_2010_hist)
    change_pct = (sector_change / len(bh_2010_hist)) * 100 if len(bh_2010_hist) > 0 else 0
    
    print(f"   • Net change: {sector_change:+,} sectors ({change_pct:+.1f}%)")
    
    # Analyze formation types
    print(f"\n🔄 Formation type distribution (2022):")
    formation_counts = bh_2022_hist['FRM_2022_DIVULGAÇÃO'].value_counts().head(10)
    
    # Formation type mapping
    formation_types = {
    # Codes 1xx: Manutenção (Maintenance)
    111: 'Setor mantido, manteve a subordenação municipal e distrital, manteve a situação',
    112: 'Setor mantido, manteve a subordenação municipal e distrital, alterou a situação',
    113: 'Setor mantido, manteve a subordinação municipal, alterou subordinação distrital ou subdistrital, manteve a situação',
    114: 'Setor mantido, manteve a subordenação municipal, alterou subordinação distrital ou subdistrital, alterou a situação',
    115: 'Setor mantido, alterou a subordinação municipal, manteve a situação',
    116: 'Setor mantido, alterou a subordinação municipal, alterou a situação',
    161: 'Setor mantido com ajuste de geometria, manteve a subordinação e manteve a situação',
    162: 'Setor mantido com ajuste de geometria, manteve a subordinação e alterou a situação',
    163: 'Setor mantido com ajuste de geometria, alterou a subordinação distrital ou subdistrital e manteve a situação',
    164: 'Setor mantido com ajuste de geometria, alterou a subordinação distrital ou subdistrital e alterou a situação',
    165: 'Setor mantido com ajuste de geometria, alterou a subordenação municipal e manteve a situação',
    166: 'Setor mantido com ajuste de geometria, alterou a subordenação municipal e alterou a situação',

    # Codes 2xx: Subdivisão/Divisão (Division)
    221: 'Subdivisão do setor por critério quantitativo, manteve a subordinação, manteve a situação',
    222: 'Subdivisão do setor por critério quantitativo, manteve a subordinação, alterou a situação',
    223: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por critério quantitativo de unidade de coleta com alteração de Distrito',
    224: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por critério quantitativo de unidade de coleta com alteração de Distrito e situação',
    225: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por critério quantitativo de unidade de coleta com alteração de Município e Distrito',
    231: 'Subdivisão do setor por critério de tamanho, manteve a subordinação, manteve a situação',
    232: 'Subdivisão do setor por critério de tamanho, manteve a subordinação, alterou a situação',
    233: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por área superior a 500 km² com alteração de Distrito',
    234: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por área superior a 500 km² com alteração de Distrito e Situação',
    241: 'Subdivisão do setor por critério de estrutura territorial, manteve a subordinação, manteve a situação',
    242: 'Subdivisão do setor por critério de estrutura territorial, manteve a subordinação, alterou a situação',
    243: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por limite de área de apuração com alteração de Distrito',
    244: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por limite de área de apuração com alteração de Distrito e Situação',
    245: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por limite de área de apuração com alteração de Município e Distrito',
    246: 'Área do setor constituída de por parte do setor do ano anterior, subdividido por limite de área de apuração com alteração de Município, Distrito e Situação',
    251: 'Subdivisão do setor por outro critério, manteve a subordinação, manteve a situação',
    252: 'Subdivisão do setor por outro critério, manteve a subordinação, alterou a situação',
    253: 'Subdivisão do setor por outro critério, alterou a subordinação distrital ou subdistrital, manteve a situação',
    254: 'Subdivisão do setor por outro critério, alterou a subordinação distrital ou subdistrital, alterou a situação',
    255: 'Subdivisão do setor por outro critério, alterou a subordinação municipal, manteve a situação',
    256: 'Subdivisão do setor por outro critério, alterou a subordinação municipal, alterou a situação',

    # Codes 3xx: Agregação (Aggregation)
    371: 'Agregação de setor com domicílio, manteve a subordinação, manteve a situação',
    372: 'Agregação de setor com domicílio, manteve a subordinação, alterou a situação',
    373: 'Área do setor constituída pela junção de setores inteiros com domicílios com alteração do Distrito',
    374: 'Área do setor constituída pela junção de setores inteiros com domicílios com alteração do Distrito e Situação',
    375: 'Área do setor constituída pela junção de setores inteiros com domicílios com alteração do Município e do Distrito',
    376: 'Área do setor constituída pela junção de setores inteiros com domicílios com alteração do Município, do Distrito e da Situação',
    391: 'Agregação de setor sem domicílio, manteve a subordinação, manteve a situação',
    392: 'Agregação de setor sem domicílio, manteve a subordinação, alterou a situação',
    394: 'Área do setor constituída pela junção de setores inteiros sem domicílios com alteração do Distrito e da Situação',
    395: 'Área do setor constituída pela junção de setores inteiros sem domicílios com alteração do Município e do Distrito'
}
    
    for code, count in formation_counts.items():
        desc = formation_types.get(code, f'Formation Code {code}')
        pct = (count / len(bh_2022_hist)) * 100
        print(f"   • {code}-{desc}: {count:,} ({pct:.1f}%)")
    
    # Show stability analysis
    unchanged_sectors = len(bh_2022_hist[bh_2022_hist['FRM_2022_DIVULGAÇÃO'] == 111])
    stability_rate = (unchanged_sectors / len(bh_2022_hist)) * 100
    
    print(f"\n✨ Geographic stability assessment:")
    print(f"   🔒 Unchanged sectors: {unchanged_sectors:,} ({stability_rate:.1f}%)")
    print(f"   🔄 Modified sectors: {len(bh_2022_hist) - unchanged_sectors:,} ({100-stability_rate:.1f}%)")
    
    stability_level = "Excellent" if stability_rate >= 85 else "Good" if stability_rate >= 70 else "Moderate"
    print(f"   📊 Stability level: {stability_level} (for longitudinal research)")
    
    # Store for visualization
    bh_formation_data = bh_2022_hist.copy()
    
else:
    print("⚠️ Cannot analyze - evolution data not available")


## 3.3 Formation Types Visualization



In [ ]:
if df_evolution is not None and len(bh_2022_hist) > 0:
    # Create comprehensive visualization of formation types
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Pie chart - Formation types distribution
    formation_counts = bh_2022_hist['FRM_2022_DIVULGAÇÃO'].value_counts()
    formation_labels = [f'Code {code}' for code in formation_counts.index]
    
    colors = ['#2E8B57', '#FF6B35', '#F7931E', '#FFD23F', '#06FFA5', '#4ECDC4', '#45B7D1', '#96CEB4', '#FF9999']
    
    wedges, texts, autotexts = ax1.pie(formation_counts.values, labels=formation_labels, 
                                      autopct='%1.1f%%', startangle=90, 
                                      colors=colors[:len(formation_counts)])
    ax1.set_title('Census Sectors Formation Types\nBelo Horizonte (2010→2022)', 
                  fontsize=12, fontweight='bold', pad=20)
    
    # Make percentage text bold and larger
    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_fontsize(9)
    
    # 2. Bar chart - Count by formation type
    top_formations = formation_counts.head(8)
    bars = ax2.bar(range(len(top_formations)), top_formations.values, 
                   color=colors[:len(top_formations)], alpha=0.8)
    
    ax2.set_xlabel('Formation Type', fontweight='bold')
    ax2.set_ylabel('Number of Sectors', fontweight='bold')
    ax2.set_title('Sector Count by Formation Type', fontsize=12, fontweight='bold')
    ax2.set_xticks(range(len(top_formations)))
    ax2.set_xticklabels([f'Code {code}' for code in top_formations.index], 
                        rotation=45, ha='right', fontsize=9)
    
    # Add value labels on bars
    for bar, value in zip(bars, top_formations.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
                f'{value:,}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Stability analysis - Unchanged vs Modified
    stability_data = {
        'Unchanged\n(Mantido)': unchanged_sectors,
        'Modified\n(All Changes)': len(bh_2022_hist) - unchanged_sectors
    }
    
    stability_colors = ['#2E8B57', '#FF6B35']
    bars3 = ax3.bar(stability_data.keys(), stability_data.values(), 
                    color=stability_colors, alpha=0.8, width=0.6)
    
    ax3.set_ylabel('Number of Sectors', fontweight='bold')
    ax3.set_title('Geographic Stability Analysis\n(2010→2022)', fontsize=12, fontweight='bold')
    
    # Add percentage labels
    total_sectors = len(bh_2022_hist)
    for bar, (label, value) in zip(bars3, stability_data.items()):
        pct = (value / total_sectors) * 100
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{value:,}\n({pct:.1f}%)', ha='center', va='bottom', 
                fontweight='bold', fontsize=10)
    
    # 4. Evolution timeline concept
    years = ['2010', '2022']
    sector_counts = [len(bh_2010_hist), len(bh_2022_hist)]
    
    ax4.plot(years, sector_counts, marker='o', linewidth=3, markersize=10, 
             color='#2E8B57', markerfacecolor='#FF6B35', markeredgewidth=2)
    ax4.fill_between(years, sector_counts, alpha=0.3, color='#2E8B57')
    
    ax4.set_ylabel('Number of Sectors', fontweight='bold')
    ax4.set_title('Sector Count Evolution\nBelo Horizonte', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # Add value labels
    for year, count in zip(years, sector_counts):
        ax4.annotate(f'{count:,}', (year, count), textcoords="offset points", 
                    xytext=(0,10), ha='center', fontweight='bold', fontsize=11)
    
    # Add change annotation
    change = sector_counts[1] - sector_counts[0]
    ax4.annotate(f'Change: {change:+,}\n({change/sector_counts[0]*100:+.1f}%)', 
                xy=(1, sector_counts[1]), xytext=(0.5, max(sector_counts) * 0.8),
                ha='center', fontweight='bold', fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor='yellow', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Visualization Summary:")
    print(f"   • Total sectors analyzed: {len(bh_2022_hist):,}")
    print(f"   • Most common: {formation_types.get(formation_counts.index[0], 'Unknown')} ({formation_counts.iloc[0]:,} sectors)")
    print(f"   • Stability rate: {stability_rate:.1f}% unchanged from 2010")
    print(f"   • Evolution: {len(bh_2010_hist):,} → {len(bh_2022_hist):,} sectors")
    
else:
    print("⚠️ Cannot create visualization - data not available")


## 3.4 Cross-Reference with Current Census Data


In [ ]:
# Cross-reference historical evolution with current notebook's 2022 census data
if df_evolution is not None and 'belo_horizonte_2022' in locals():
    print("🔄 Cross-Reference Analysis: Historical Evolution ↔ Current Census Data")
    print("=" * 75)
    
    # Get sector codes from current notebook data
    if 'CD_SETOR' in belo_horizonte_2022.columns:
        current_sectors = set(belo_horizonte_2022['CD_SETOR'].astype(str))
        sector_col = 'CD_SETOR'
    else:
        # Try first column as fallback
        sector_col = belo_horizonte_2022.columns[0]
        current_sectors = set(belo_horizonte_2022[sector_col].astype(str))
        print(f"   📝 Using column '{sector_col}' for sector identification")
    
    # Get sectors from historical data
    historical_sectors = set(bh_2022_hist['GEOCODIGO_2022_DIVULGAÇÃO'].astype(str))
    
    print(f"\n📊 Dataset comparison:")
    print(f"   • Current notebook sectors: {len(current_sectors):,}")
    print(f"   • Historical evolution sectors: {len(historical_sectors):,}")
    
    # Find matches
    matching_sectors = current_sectors.intersection(historical_sectors)
    match_rate = (len(matching_sectors) / len(current_sectors)) * 100 if len(current_sectors) > 0 else 0
    
    print(f"   • Matching sectors: {len(matching_sectors):,}")
    print(f"   • Match rate: {match_rate:.1f}%")
    
    if len(matching_sectors) > 0:
        print(f"\n✅ Cross-reference successful!")
        print(f"   📝 Sample matching sectors: {list(matching_sectors)[:5]}")
        
        # Evolution analysis for matching sectors
        matching_evolution = bh_2022_hist[
            bh_2022_hist['GEOCODIGO_2022_DIVULGAÇÃO'].astype(str).isin(matching_sectors)
        ]
        
        print(f"\n📈 Evolution analysis for {len(matching_evolution):,} matching sectors:")
        
        evolution_summary = matching_evolution['FRM_2022_DIVULGAÇÃO'].value_counts().head(6)
        for code, count in evolution_summary.items():
            desc = formation_types.get(code, f'Code {code}')
            pct = (count / len(matching_evolution)) * 100
            print(f"   • {desc}: {count:,} ({pct:.1f}%)")
        
        # Research implications
        unchanged_in_sample = len(matching_evolution[matching_evolution['FRM_2022_DIVULGAÇÃO'] == 111])
        continuity_rate = (unchanged_in_sample / len(matching_evolution)) * 100
        
        print(f"\n🎯 Research implications:")
        print(f"   🔒 Geographic continuity: {continuity_rate:.1f}% of study sectors unchanged")
        print(f"   📊 Data reliability: {'Excellent' if continuity_rate >= 85 else 'Good' if continuity_rate >= 70 else 'Moderate'}")
        print(f"   🔬 Longitudinal validity: {'High' if match_rate >= 95 else 'Medium' if match_rate >= 80 else 'Limited'}")
        
        # Store matching data for further analysis
        sectors_with_evolution = matching_evolution.copy()
        
    else:
        print(f"\n⚠️ No sector matches found!")
        print(f"   This might indicate different coding systems or data sources")
        
elif 'belo_horizonte_2022' not in locals():
    print("⚠️ Current census data not available for cross-reference")
    print("   Please run the 2022 census data loading sections first")
else:
    print("⚠️ Historical evolution data not available")


## 3.5 Formation Types Deep Dive Analysis


In [ ]:
if df_evolution is not None and len(bh_2022_hist) > 0:
    print("🔍 Formation Types Deep Dive Analysis")
    print("=" * 50)
    
    # Detailed formation analysis
    formation_analysis = bh_2022_hist.groupby('FRM_2022_DIVULGAÇÃO').agg({
        'GEOCODIGO_2022_DIVULGAÇÃO': 'count',
        'GEOCODIGO_2010': lambda x: x.notna().sum()  # Count non-null 2010 codes
    }).rename(columns={
        'GEOCODIGO_2022_DIVULGAÇÃO': 'Count_2022',
        'GEOCODIGO_2010': 'Has_2010_Code'
    })
    
    formation_analysis['Percentage'] = (formation_analysis['Count_2022'] / len(bh_2022_hist)) * 100
    formation_analysis['Description'] = formation_analysis.index.map(
        lambda x: formation_types.get(x, f'Code {x}')
    )
    
    # Sort by count
    formation_analysis = formation_analysis.sort_values('Count_2022', ascending=False)
    
    print(f"📊 Formation types breakdown:")
    print(f"{'Code':<5} {'Description':<25} {'Count':<8} {'%':<6} {'Has 2010':<10}")
    print("-" * 60)
    
    for code, row in formation_analysis.head(10).iterrows():
        print(f"{code:<5} {row['Description'][:24]:<25} {row['Count_2022']:<8,} "
              f"{row['Percentage']:<6.1f} {row['Has_2010_Code']:<10,}")
    
    # Create detailed visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Formation types horizontal bar chart
    top_8 = formation_analysis.head(8)
    y_pos = range(len(top_8))
    
    bars = ax1.barh(y_pos, top_8['Count_2022'], color=colors[:len(top_8)], alpha=0.8)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([desc[:20] + '...' if len(desc) > 20 else desc 
                        for desc in top_8['Description']], fontsize=10)
    ax1.set_xlabel('Number of Sectors', fontweight='bold')
    ax1.set_title('Formation Types Distribution\n(Top 8 Categories)', fontweight='bold')
    
    # Add value labels
    for bar, value in zip(bars, top_8['Count_2022']):
        ax1.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                f'{value:,}', va='center', ha='left', fontweight='bold')
    
    # Comparison of sectors with/without 2010 codes
    continuity_data = {
        'Has 2010 Reference': formation_analysis['Has_2010_Code'].sum(),
        'No 2010 Reference': (formation_analysis['Count_2022'] - formation_analysis['Has_2010_Code']).sum()
    }
    
    wedges, texts, autotexts = ax2.pie(continuity_data.values(), labels=continuity_data.keys(),
                                      autopct='%1.1f%%', startangle=90,
                                      colors=['#2E8B57', '#FF6B35'])
    ax2.set_title('Temporal Continuity\n(2010 Reference Availability)', fontweight='bold')
    
    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)
    
    plt.tight_layout()
    plt.show()
    
    # Key insights
    print(f"\n💡 Key insights:")
    most_common = formation_analysis.index[0]
    most_common_desc = formation_analysis.loc[most_common, 'Description']
    most_common_count = formation_analysis.loc[most_common, 'Count_2022']
    most_common_pct = formation_analysis.loc[most_common, 'Percentage']
    
    print(f"   • Most common formation: {most_common_desc} ({most_common_count:,} sectors, {most_common_pct:.1f}%)")
    
    total_with_2010 = formation_analysis['Has_2010_Code'].sum()
    continuity_pct = (total_with_2010 / len(bh_2022_hist)) * 100
    print(f"   • Temporal continuity: {total_with_2010:,} sectors ({continuity_pct:.1f}%) have 2010 references")
    
    new_sectors = formation_analysis.loc[formation_analysis.index == 221, 'Count_2022'].sum() if 221 in formation_analysis.index else 0
    print(f"   • New sectors created: {new_sectors:,} (enhanced spatial resolution)")
    
    subdivided = formation_analysis.loc[formation_analysis.index == 161, 'Count_2022'].sum() if 161 in formation_analysis.index else 0
    print(f"   • Subdivided sectors: {subdivided:,} (improved granularity)")
    
else:
    print("⚠️ Cannot perform deep dive analysis - data not available")


## 3.6 Research Implications Summary


In [ ]:
# Comprehensive summary of historical evolution analysis
if df_evolution is not None:
    print("🎯 RESEARCH IMPLICATIONS SUMMARY")
    print("=" * 60)
    print("Historical Census Sectors Evolution Analysis - Final Report")
    print("=" * 60)
    
    # Calculate key metrics
    total_bh_sectors_2022 = len(bh_2022_hist)
    total_bh_sectors_2010 = len(bh_2010_hist)
    sector_change = total_bh_sectors_2022 - total_bh_sectors_2010
    
    unchanged_sectors = len(bh_2022_hist[bh_2022_hist['FRM_2022_DIVULGAÇÃO'] == 111])
    stability_rate = (unchanged_sectors / total_bh_sectors_2022) * 100
    
    # Formation statistics
    formation_stats = bh_2022_hist['FRM_2022_DIVULGAÇÃO'].value_counts()
    
    print(f"\n📊 QUANTITATIVE SUMMARY:")
    print(f"   • Evolution Period: 2010 → 2022 (12 years)")
    print(f"   • Geographic Scope: Belo Horizonte Municipality")
    print(f"   • Sector Count Change: {total_bh_sectors_2010:,} → {total_bh_sectors_2022:,} ({sector_change:+,})")
    print(f"   • Geographic Stability: {stability_rate:.1f}% sectors unchanged")
    print(f"   • Data Coverage: {len(df_evolution):,} national records analyzed")
    
    print(f"\n🔬 EPIDEMIOLOGICAL RESEARCH IMPLICATIONS:")
    
    print(f"\n   1️⃣ LONGITUDINAL ANALYSIS VALIDITY:")
    if stability_rate >= 90:
        validity = "Excellent"
        recommendation = "Ideal for temporal trend analysis"
    elif stability_rate >= 80:
        validity = "Very Good"  
        recommendation = "Suitable for most longitudinal studies"
    elif stability_rate >= 70:
        validity = "Good"
        recommendation = "Acceptable with sector mapping considerations"
    else:
        validity = "Limited"
        recommendation = "Requires careful sector reconciliation"
    
    print(f"      • Geographic Continuity: {validity}")
    print(f"      • Recommendation: {recommendation}")
    print(f"      • Unchanged Sectors Available: {unchanged_sectors:,}")
    
    print(f"\n   2️⃣ SPATIAL RESOLUTION ENHANCEMENT:")
    subdivided = formation_stats.get(161, 0)  # Subdivided sectors
    new_sectors = formation_stats.get(221, 0)  # New sectors
    enhanced_resolution = subdivided + new_sectors
    
    print(f"      • Enhanced Granularity: {enhanced_resolution:,} sectors")
    print(f"      • Subdivided Areas: {subdivided:,} (better population targeting)")
    print(f"      • Completely New Areas: {new_sectors:,} (expanded coverage)")
    print(f"      • Resolution Improvement: {(enhanced_resolution/total_bh_sectors_2022)*100:.1f}% of sectors")
    
    print(f"\n   3️⃣ OVITRAPS PLACEMENT OPTIMIZATION:")
    print(f"      • Stable Locations: {unchanged_sectors:,} sectors for consistent monitoring")
    print(f"      • New Opportunities: {enhanced_resolution:,} sectors for expanded coverage")
    print(f"      • Risk Assessment: Improved precision with {total_bh_sectors_2022:,} total areas")
    print(f"      • Population Weighting: Enhanced demographic accuracy available")
    
    print(f"\n   4️⃣ DATA INTEGRATION CAPACITY:")
    if 'sectors_with_evolution' in locals():
        integration_rate = (len(sectors_with_evolution) / total_bh_sectors_2022) * 100
        print(f"      • Current Data Compatibility: {integration_rate:.1f}% sectors matched")
    print(f"      • Historical Linkage: Complete 2010-2022 evolution tracking")
    print(f"      • Future Extensibility: Framework ready for 2030 census integration")
    
    # Create summary visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Evolution timeline
    years = ['2010', '2022']
    counts = [total_bh_sectors_2010, total_bh_sectors_2022]
    ax1.plot(years, counts, 'o-', linewidth=3, markersize=10, color='#2E8B57')
    ax1.fill_between(years, counts, alpha=0.3, color='#2E8B57')
    ax1.set_title('Sector Count Evolution', fontweight='bold')
    ax1.set_ylabel('Number of Sectors')
    for year, count in zip(years, counts):
        ax1.annotate(f'{count:,}', (year, count), textcoords="offset points", 
                    xytext=(0,10), ha='center', fontweight='bold')
    
    # 2. Stability overview
    stability_data = {'Unchanged': unchanged_sectors, 'Modified': total_bh_sectors_2022 - unchanged_sectors}
    ax2.pie(stability_data.values(), labels=stability_data.keys(), autopct='%1.1f%%',
            colors=['#2E8B57', '#FF6B35'], startangle=90)
    ax2.set_title('Geographic Stability', fontweight='bold')
    
    # 3. Resolution enhancement
    resolution_data = {
        'Original': total_bh_sectors_2022 - enhanced_resolution,
        'Enhanced': enhanced_resolution
    }
    ax3.bar(resolution_data.keys(), resolution_data.values(), 
            color=['#4ECDC4', '#F7931E'], alpha=0.8)
    ax3.set_title('Spatial Resolution Enhancement', fontweight='bold')
    ax3.set_ylabel('Number of Sectors')
    for i, (key, value) in enumerate(resolution_data.items()):
        ax3.text(i, value + 50, f'{value:,}', ha='center', va='bottom', fontweight='bold')
    
    # 4. Research readiness score
    readiness_metrics = {
        'Stability': min(100, stability_rate),
        'Coverage': min(100, (total_bh_sectors_2022/5000) * 100),  # Relative to expected coverage
        'Resolution': min(100, (enhanced_resolution/1000) * 100),   # Relative to enhancement target
        'Integration': 95 if 'sectors_with_evolution' in locals() else 85  # Data compatibility
    }
    
    metrics = list(readiness_metrics.keys())
    scores = list(readiness_metrics.values())
    
    bars = ax4.bar(metrics, scores, color=['#2E8B57', '#45B7D1', '#F7931E', '#FF6B35'], alpha=0.8)
    ax4.set_title('Research Readiness Score', fontweight='bold')
    ax4.set_ylabel('Score (%)')
    ax4.set_ylim(0, 100)
    ax4.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='Target Threshold')
    
    for bar, score in zip(bars, scores):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{score:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ ANALYSIS COMPLETE!")
    print(f"   📋 Historical evolution data successfully analyzed")
    print(f"   📊 {total_bh_sectors_2022:,} Belo Horizonte sectors characterized")
    print(f"   🎯 {stability_rate:.1f}% geographic stability confirmed")
    print(f"   🔬 Research framework validated for longitudinal epidemiological studies")
    print(f"   📈 Enhanced spatial resolution available for improved ovitraps targeting")

else:
    print("⚠️ Analysis incomplete - historical evolution data not available")

## 3.7 Census Regions Linkage Analysis (2010 ↔ 2022)
Analyzing the presence of census sectors from both 2010 and 2022 in the historical evolution data, and creating a mapping structure to link sectors across both censuses.


In [ ]:
# Check presence of 2010 and 2022 census sectors in historical evolution data
if df_evolution is not None:
    print("🔗 Census Regions Linkage Analysis")
    print("=" * 70)
    
    # Extract sector codes from our census dataframes
    print("📋 Extracting sector codes from census data...")
    
    # 2010 sectors from bh_setores_gdf
    if 'bh_setores_gdf' in locals() and bh_setores_gdf is not None and 'CD_GEOCODI' in bh_setores_gdf.columns:
        sectors_2010 = set(bh_setores_gdf['CD_GEOCODI'].astype(str))
        print(f"   • 2010 sectors (bh_setores_gdf): {len(sectors_2010):,}")
    else:
        sectors_2010 = set()
        print(f"   ⚠️ 2010 data not found (bh_setores_gdf.CD_GEOCODI)")
    
    # 2022 sectors from sectors_with_data_2022
    if 'sectors_with_data_2022' in locals() and sectors_with_data_2022 is not None and 'CD_SETOR' in sectors_with_data_2022.columns:
        sectors_2022 = set(sectors_with_data_2022['CD_SETOR'].astype(str))
        print(f"   • 2022 sectors (sectors_with_data_2022): {len(sectors_2022):,}")
    else:
        sectors_2022 = set()
        print(f"   ⚠️ 2022 data not found (sectors_with_data_2022.CD_SETOR)")
    
    # Get sectors from historical evolution CSV
    historical_2010 = set(df_evolution['GEOCODIGO_2010'].dropna().astype(str))
    historical_2022 = set(df_evolution['GEOCODIGO_2022_DIVULGAÇÃO'].dropna().astype(str))
    
    print(f"   • Historical CSV 2010 sectors: {len(historical_2010):,}")
    print(f"   • Historical CSV 2022 sectors: {len(historical_2022):,}")
    
    # Check coverage for 2010
    if len(sectors_2010) > 0:
        print(f"\n🔍 2010 Census Coverage Analysis:")
        found_2010 = sectors_2010.intersection(historical_2010)
        missing_2010 = sectors_2010 - historical_2010
        
        coverage_2010 = (len(found_2010) / len(sectors_2010)) * 100
        print(f"   • Found in CSV: {len(found_2010):,} sectors ({coverage_2010:.1f}%)")
        print(f"   • Missing from CSV: {len(missing_2010):,} sectors ({100-coverage_2010:.1f}%)")
        
        if len(missing_2010) > 0 and len(missing_2010) <= 10:
            print(f"   • Missing sectors: {list(missing_2010)}")
        elif len(missing_2010) > 10:
            print(f"   • Sample missing sectors: {list(missing_2010)[:10]}")
    
    # Check coverage for 2022
    if len(sectors_2022) > 0:
        print(f"\n🔍 2022 Census Coverage Analysis:")
        found_2022 = sectors_2022.intersection(historical_2022)
        missing_2022 = sectors_2022 - historical_2022
        
        coverage_2022 = (len(found_2022) / len(sectors_2022)) * 100
        print(f"   • Found in CSV: {len(found_2022):,} sectors ({coverage_2022:.1f}%)")
        print(f"   • Missing from CSV: {len(missing_2022):,} sectors ({100-coverage_2022:.1f}%)")
        
        if len(missing_2022) > 0 and len(missing_2022) <= 10:
            print(f"   • Missing sectors: {list(missing_2022)}")
        elif len(missing_2022) > 10:
            print(f"   • Sample missing sectors: {list(missing_2022)[:10]}")
    
    print(f"\n{'='*70}")
    
else:
    print("⚠️ Historical evolution data not available")


In [ ]:

# Create linkage structure between 2010 and 2022 sectors
if df_evolution is not None and len(sectors_2010) > 0 and len(sectors_2022) > 0:
    print("🔗 Creating 2010 ↔ 2022 Sector Linkage Structure")
    print("=" * 70)
    
    # Filter historical data for Belo Horizonte
    bh_evolution = df_evolution[
        (df_evolution['GEOCODIGO_2010'].astype(str).str.startswith('3106200')) |
        (df_evolution['GEOCODIGO_2022_DIVULGAÇÃO'].astype(str).str.startswith('3106200'))
    ].copy()
    
    print(f"📊 BH evolution records: {len(bh_evolution):,}")
    
    # Create comprehensive linkage dataframe
    linkage_data = []
    
    for _, row in bh_evolution.iterrows():
        sector_2010 = str(row['GEOCODIGO_2010']) if pd.notna(row['GEOCODIGO_2010']) else None
        sector_2022 = str(row['GEOCODIGO_2022_DIVULGAÇÃO']) if pd.notna(row['GEOCODIGO_2022_DIVULGAÇÃO']) else None
        formation_code = row['FRM_2022_DIVULGAÇÃO']
        formation_desc = formation_types.get(formation_code, f'Code {formation_code}')
        
        # Check if sectors are in our datasets
        in_2010_data = sector_2010 in sectors_2010 if sector_2010 else False
        in_2022_data = sector_2022 in sectors_2022 if sector_2022 else False
        
        linkage_data.append({
            'SETOR_2010': sector_2010,
            'SETOR_2022': sector_2022,
            'FORMATION_CODE': formation_code,
            'FORMATION_TYPE': formation_desc,
            'IN_2010_DATASET': in_2010_data,
            'IN_2022_DATASET': in_2022_data,
            'COMPLETE_LINK': in_2010_data and in_2022_data
        })
    
    # Create DataFrame
    sector_linkage_df = pd.DataFrame(linkage_data)
    
    print(f"\n✅ Linkage structure created!")
    print(f"   • Total linkage records: {len(sector_linkage_df):,}")
    print(f"   • Records with 2010 data: {sector_linkage_df['IN_2010_DATASET'].sum():,}")
    print(f"   • Records with 2022 data: {sector_linkage_df['IN_2022_DATASET'].sum():,}")
    print(f"   • Complete links (both censuses): {sector_linkage_df['COMPLETE_LINK'].sum():,}")
    
    # Show linkage quality by formation type
    print(f"\n📊 Linkage quality by formation type:")
    linkage_quality = sector_linkage_df.groupby('FORMATION_TYPE').agg({
        'COMPLETE_LINK': ['sum', 'count']
    })
    linkage_quality.columns = ['Complete_Links', 'Total']
    linkage_quality['Link_Rate_%'] = (linkage_quality['Complete_Links'] / linkage_quality['Total'] * 100).round(1)
    linkage_quality = linkage_quality.sort_values('Total', ascending=False)
    
    print(linkage_quality.head(10).to_string())
    
    # Display sample linkages
    print(f"\n📋 Sample linkage records:")
    display(sector_linkage_df[sector_linkage_df['COMPLETE_LINK'] == True].head(10))
    
else:
    print("⚠️ Cannot create linkage - missing required data")
    if len(sectors_2010) == 0:
        print("   • 2010 sector data not available")
    if len(sectors_2022) == 0:
        print("   • 2022 sector data not available")



In [ ]:
# Create specific mapping dictionaries for easy lookup
if 'sector_linkage_df' in locals() and len(sector_linkage_df) > 0:
    print("🗺️ Creating Mapping Dictionaries")
    print("=" * 50)
    
    # Forward mapping: 2010 → 2022
    map_2010_to_2022 = {}
    for _, row in sector_linkage_df[sector_linkage_df['SETOR_2010'].notna()].iterrows():
        sector_2010 = row['SETOR_2010']
        sector_2022 = row['SETOR_2022']
        formation = row['FORMATION_TYPE']
        
        if sector_2010 not in map_2010_to_2022:
            map_2010_to_2022[sector_2010] = []
        
        map_2010_to_2022[sector_2010].append({
            'sector_2022': sector_2022,
            'formation_type': formation,
            'formation_code': row['FORMATION_CODE']
        })
    
    # Reverse mapping: 2022 → 2010
    map_2022_to_2010 = {}
    for _, row in sector_linkage_df[sector_linkage_df['SETOR_2022'].notna()].iterrows():
        sector_2022 = row['SETOR_2022']
        sector_2010 = row['SETOR_2010']
        formation = row['FORMATION_TYPE']
        
        if sector_2022 not in map_2022_to_2010:
            map_2022_to_2010[sector_2022] = []
        
        map_2022_to_2010[sector_2022].append({
            'sector_2010': sector_2010,
            'formation_type': formation,
            'formation_code': row['FORMATION_CODE']
        })
    
    print(f"✅ Mapping dictionaries created!")
    print(f"   • 2010→2022 mappings: {len(map_2010_to_2022):,} sectors")
    print(f"   • 2022→2010 mappings: {len(map_2022_to_2010):,} sectors")
    
    # Analyze mapping complexity
    multi_2010 = sum(1 for v in map_2010_to_2022.values() if len(v) > 1)
    multi_2022 = sum(1 for v in map_2022_to_2010.values() if len(v) > 1)
    
    print(f"\n📊 Mapping complexity:")
    print(f"   • 2010 sectors → multiple 2022 sectors: {multi_2010:,} (subdivisions)")
    print(f"   • 2022 sectors → multiple 2010 sectors: {multi_2022:,} (aggregations)")
    
    # Show examples
    print(f"\n📝 Example mappings:")
    
    # Simple 1:1 mapping
    simple_mappings = [k for k, v in map_2010_to_2022.items() if len(v) == 1]
    if simple_mappings:
        example_2010 = simple_mappings[0]
        example_2022 = map_2010_to_2022[example_2010][0]
        print(f"\n   1:1 Mapping (Unchanged):")
        print(f"   {example_2010} → {example_2022['sector_2022']}")
        print(f"   Formation: {example_2022['formation_type']}")
    
    # Complex mapping (subdivision)
    complex_mappings = [k for k, v in map_2010_to_2022.items() if len(v) > 1]
    if complex_mappings:
        example_2010 = complex_mappings[0]
        mappings = map_2010_to_2022[example_2010]
        print(f"\n   1:N Mapping (Subdivision):")
        print(f"   {example_2010} →")
        for m in mappings[:3]:
            print(f"      • {m['sector_2022']} ({m['formation_type']})")
        if len(mappings) > 3:
            print(f"      ... and {len(mappings)-3} more")
    
    print(f"\n💾 Variables created:")
    print(f"   • sector_linkage_df: Complete linkage DataFrame")
    print(f"   • map_2010_to_2022: Dictionary for forward lookup")
    print(f"   • map_2022_to_2010: Dictionary for reverse lookup")
    
else:
    print("⚠️ Cannot create mapping dictionaries - linkage structure not available")



In [ ]:
# Visualize the linkage relationships
if 'sector_linkage_df' in locals() and len(sector_linkage_df) > 0:
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Coverage by census
    coverage_data = {
        '2010 Data\nAvailable': sector_linkage_df['IN_2010_DATASET'].sum(),
        '2010 Data\nMissing': (~sector_linkage_df['IN_2010_DATASET']).sum(),
        '2022 Data\nAvailable': sector_linkage_df['IN_2022_DATASET'].sum(),
        '2022 Data\nMissing': (~sector_linkage_df['IN_2022_DATASET']).sum()
    }
    
    colors_coverage = ['#2E8B57', '#FF6B35'] * 2
    bars1 = ax1.bar(range(len(coverage_data)), coverage_data.values(), 
                    color=colors_coverage, alpha=0.8)
    ax1.set_xticks(range(len(coverage_data)))
    ax1.set_xticklabels(coverage_data.keys(), fontsize=9)
    ax1.set_ylabel('Number of Sectors', fontweight='bold')
    ax1.set_title('Data Availability by Census Year', fontweight='bold')
    
    for bar, value in zip(bars1, coverage_data.values()):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{value:,}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Linkage completeness
    linkage_status = {
        'Complete Link\n(Both Censuses)': sector_linkage_df['COMPLETE_LINK'].sum(),
        'Partial Link\n(One Census)': (~sector_linkage_df['COMPLETE_LINK']).sum()
    }
    
    wedges, texts, autotexts = ax2.pie(linkage_status.values(), labels=linkage_status.keys(),
                                      autopct='%1.1f%%', startangle=90,
                                      colors=['#2E8B57', '#FFD23F'])
    ax2.set_title('Linkage Completeness', fontweight='bold')
    
    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)
    
    # 3. Mapping complexity distribution
    mapping_complexity = {
        '1:1\n(Simple)': len([k for k, v in map_2010_to_2022.items() if len(v) == 1]),
        '1:N\n(Subdivision)': len([k for k, v in map_2010_to_2022.items() if len(v) > 1])
    }
    
    bars3 = ax3.bar(mapping_complexity.keys(), mapping_complexity.values(),
                    color=['#4ECDC4', '#F7931E'], alpha=0.8)
    ax3.set_ylabel('Number of 2010 Sectors', fontweight='bold')
    ax3.set_title('Mapping Complexity (2010→2022)', fontweight='bold')
    
    for bar, value in zip(bars3, mapping_complexity.values()):
        pct = (value / sum(mapping_complexity.values())) * 100
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{value:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
    
    # 4. Formation type distribution for linked sectors
    linked_sectors = sector_linkage_df[sector_linkage_df['COMPLETE_LINK'] == True]
    formation_dist = linked_sectors['FORMATION_TYPE'].value_counts().head(6)
    
    bars4 = ax4.barh(range(len(formation_dist)), formation_dist.values,
                     color=colors[:len(formation_dist)], alpha=0.8)
    ax4.set_yticks(range(len(formation_dist)))
    ax4.set_yticklabels([f[:20] for f in formation_dist.index], fontsize=9)
    ax4.set_xlabel('Number of Complete Links', fontweight='bold')
    ax4.set_title('Formation Types (Sectors with Complete Links)', fontweight='bold')
    
    for bar, value in zip(bars4, formation_dist.values):
        ax4.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f'{value:,}', va='center', ha='left', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Linkage Visualization Summary:")
    print(f"   • Total records analyzed: {len(sector_linkage_df):,}")
    print(f"   • Complete links: {sector_linkage_df['COMPLETE_LINK'].sum():,} ({(sector_linkage_df['COMPLETE_LINK'].sum()/len(sector_linkage_df)*100):.1f}%)")
    print(f"   • Simple mappings (1:1): {mapping_complexity['1:1\n(Simple)']:,}")
    print(f"   • Complex mappings (1:N): {mapping_complexity['1:N\n(Subdivision)']:,}")

else:
    print("⚠️ Cannot create visualization - linkage data not available")



In [ ]:
# Function to look up sector relationships
def lookup_sector_2010_to_2022(sector_2010):
    """
    Look up what happened to a 2010 sector in 2022.
    
    Parameters:
    -----------
    sector_2010 : str
        The 2010 sector code (CD_GEOCODI)
    
    Returns:
    --------
    list of dict : Information about corresponding 2022 sector(s)
    """
    if sector_2010 in map_2010_to_2022:
        return map_2010_to_2022[sector_2010]
    else:
        return None

def lookup_sector_2022_to_2010(sector_2022):
    """
    Look up the origin of a 2022 sector from 2010.
    
    Parameters:
    -----------
    sector_2022 : str
        The 2022 sector code (CD_SETOR)
    
    Returns:
    --------
    list of dict : Information about corresponding 2010 sector(s)
    """
    if sector_2022 in map_2022_to_2010:
        return map_2022_to_2010[sector_2022]
    else:
        return None

# Test the lookup functions
if 'map_2010_to_2022' in locals() and 'map_2022_to_2010' in locals():
    print("🔍 Sector Lookup Functions Created")
    print("=" * 50)
    print("Functions available:")
    print("  • lookup_sector_2010_to_2022(sector_2010)")
    print("  • lookup_sector_2022_to_2010(sector_2022)")
    
    print(f"\n📝 Example usage:")
    
    # Test with first sector from 2010
    if len(sectors_2010) > 0:
        test_sector_2010 = list(sectors_2010)[0]
        result = lookup_sector_2010_to_2022(test_sector_2010)
        
        print(f"\n2010 → 2022 lookup:")
        print(f"  Sector 2010: {test_sector_2010}")
        if result:
            print(f"  Found {len(result)} corresponding 2022 sector(s):")
            for r in result:
                print(f"    → {r['sector_2022']} ({r['formation_type']})")
        else:
            print(f"  No mapping found in evolution data")
    
    # Test with first sector from 2022
    if len(sectors_2022) > 0:
        test_sector_2022 = list(sectors_2022)[0]
        result = lookup_sector_2022_to_2010(test_sector_2022)
        
        print(f"\n2022 → 2010 lookup:")
        print(f"  Sector 2022: {test_sector_2022}")
        if result:
            print(f"  Found {len(result)} corresponding 2010 sector(s):")
            for r in result:
                print(f"    ← {r['sector_2010']} ({r['formation_type']})")
        else:
            print(f"  No mapping found in evolution data")

else:
    print("⚠️ Lookup functions not available - create mappings first")



## 3.8 Summary Statistics and Export

Final statistics and export of the linkage structure for use in epidemiological analysis.


In [ ]:
# Summary statistics and data export
if 'sector_linkage_df' in locals():
    print("📊 FINAL LINKAGE ANALYSIS SUMMARY")
    print("=" * 70)
    
    # Overall statistics
    total_records = len(sector_linkage_df)
    complete_links = sector_linkage_df['COMPLETE_LINK'].sum()
    in_2010_only = (sector_linkage_df['IN_2010_DATASET'] & ~sector_linkage_df['IN_2022_DATASET']).sum()
    in_2022_only = (~sector_linkage_df['IN_2010_DATASET'] & sector_linkage_df['IN_2022_DATASET']).sum()
    
    print(f"\n📈 Coverage Statistics:")
    print(f"   • Total evolution records: {total_records:,}")
    print(f"   • Complete links (both censuses): {complete_links:,} ({complete_links/total_records*100:.1f}%)")
    print(f"   • 2010 data only: {in_2010_only:,}")
    print(f"   • 2022 data only: {in_2022_only:,}")
    
    # Research data coverage
    if len(sectors_2010) > 0:
        coverage_2010 = (sector_linkage_df['IN_2010_DATASET'].sum() / len(sectors_2010)) * 100
        print(f"\n🔬 Research Dataset Coverage:")
        print(f"   • 2010 census sectors tracked: {sector_linkage_df['IN_2010_DATASET'].sum():,}/{len(sectors_2010):,} ({coverage_2010:.1f}%)")
    
    if len(sectors_2022) > 0:
        coverage_2022 = (sector_linkage_df['IN_2022_DATASET'].sum() / len(sectors_2022)) * 100
        print(f"   • 2022 census sectors tracked: {sector_linkage_df['IN_2022_DATASET'].sum():,}/{len(sectors_2022):,} ({coverage_2022:.1f}%)")
    
    # Mapping statistics
    print(f"\n🗺️ Mapping Structure:")
    print(f"   • 2010 sectors with forward mapping: {len(map_2010_to_2022):,}")
    print(f"   • 2022 sectors with reverse mapping: {len(map_2022_to_2010):,}")
    
    one_to_one = sum(1 for v in map_2010_to_2022.values() if len(v) == 1)
    one_to_many = sum(1 for v in map_2010_to_2022.values() if len(v) > 1)
    
    print(f"   • Simple 1:1 mappings: {one_to_one:,} ({one_to_one/len(map_2010_to_2022)*100:.1f}%)")
    print(f"   • Complex 1:N mappings: {one_to_many:,} ({one_to_many/len(map_2010_to_2022)*100:.1f}%)")
    
    # Export linkage data
    export_dir = Path("../../results/historical_analysis")
    export_dir.mkdir(exist_ok=True)
    
    # Export main linkage table
    linkage_export_path = export_dir / "sector_linkage_2010_2022.csv"
    sector_linkage_df.to_csv(linkage_export_path, index=False, encoding='utf-8')
    print(f"\n💾 Data exported:")
    print(f"   • Linkage table: {linkage_export_path.name}")
    
    # Export summary by formation type
    summary_by_formation = sector_linkage_df.groupby('FORMATION_TYPE').agg({
        'SETOR_2010': 'count',
        'COMPLETE_LINK': 'sum'
    }).rename(columns={'SETOR_2010': 'Total_Records', 'COMPLETE_LINK': 'Complete_Links'})
    summary_by_formation['Completion_Rate_%'] = (summary_by_formation['Complete_Links'] / summary_by_formation['Total_Records'] * 100).round(1)
    
    summary_export_path = export_dir / "linkage_summary_by_formation.csv"
    summary_by_formation.to_csv(summary_export_path, encoding='utf-8')
    print(f"   • Formation summary: {summary_export_path.name}")
    
    print(f"\n✅ Linkage analysis complete!")
    print(f"   Ready for longitudinal epidemiological analysis")
    print(f"   Use lookup functions to trace sector evolution")

else:
    print("⚠️ Cannot generate summary - linkage data not available")

## 3.9 Geographic Visualization: Sector Evolution Maps (2010 ↔ 2022)

Creating paired maps to visualize how census sectors evolved between 2010 and 2022, with color-coded correspondence for:
1. **Maintained sectors** - Sectors that remained unchanged
2. **Subdivided sectors** - 2010 sectors that were divided into multiple 2022 sectors
3. **Aggregated sectors** - Multiple 2010 sectors that were merged into single 2022 sectors



In [ ]:
# Prepare data for geographic visualization
if 'sector_linkage_df' in locals() and 'bh_setores_gdf' in locals() and 'sectors_with_data_2022' in locals():
    import random
    
    print("🗺️ Preparing Geographic Data for Visualization")
    print("=" * 60)
    
    # Ensure GeoDataFrames have geometry
    if 'geometry' not in bh_setores_gdf.columns:
        print("⚠️ 2010 data missing geometry - cannot create maps")
    elif 'geometry' not in sectors_with_data_2022.columns:
        print("⚠️ 2022 data missing geometry - cannot create maps")
    else:
        # Create copies
        gdf_2010 = bh_setores_gdf.copy()
        gdf_2022 = sectors_with_data_2022.copy()
        
        print(f"✅ Geographic data ready:")
        print(f"   • 2010 sectors with geometry: {len(gdf_2010):,}")
        print(f"   • 2022 sectors with geometry: {len(gdf_2022):,}")
        
        # Create sector ID columns as strings
        gdf_2010['sector_id'] = gdf_2010['CD_GEOCODI'].astype(str)
        gdf_2022['sector_id'] = gdf_2022['CD_SETOR'].astype(str)
        
        # Identify different types of evolution
        complete_links = sector_linkage_df[sector_linkage_df['COMPLETE_LINK'] == True].copy()
        
        # 1. Maintained sectors (1:1 mapping, formation code 111)
        maintained = complete_links[complete_links['FORMATION_CODE'] == 111]
        
        # 2. Subdivided sectors (1:N mapping - one 2010 → multiple 2022)
        subdivided_2010 = []
        for sector_2010 in map_2010_to_2022.keys():
            mappings = map_2010_to_2022[sector_2010]
            if len(mappings) > 1:
                subdivided_2010.append(sector_2010)
        
        subdivided = complete_links[complete_links['SETOR_2010'].isin(subdivided_2010)]
        
        # 3. Aggregated sectors (N:1 mapping - multiple 2010 → one 2022)
        aggregated_2022 = []
        for sector_2022 in map_2022_to_2010.keys():
            origins = map_2022_to_2010[sector_2022]
            if len(origins) > 1:
                aggregated_2022.append(sector_2022)
        
        aggregated = complete_links[complete_links['SETOR_2022'].isin(aggregated_2022)]
        
        print(f"\n📊 Evolution categories identified:")
        print(f"   • Maintained (unchanged): {len(maintained):,} linkages")
        print(f"   • Subdivided (1→N): {len(subdivided):,} linkages ({len(set(subdivided['SETOR_2010'])):,} unique 2010 sectors)")
        print(f"   • Aggregated (N→1): {len(aggregated):,} linkages ({len(set(aggregated['SETOR_2022'])):,} unique 2022 sectors)")
        
        # Store for mapping
        maintained_links = maintained
        subdivided_links = subdivided
        aggregated_links = aggregated

else:
    print("⚠️ Cannot prepare geographic data - missing required datasets")
    print("   Required: sector_linkage_df, bh_setores_gdf, sectors_with_data_2022")



In [ ]:
# Helper function to generate distinct colors for sector groups
def generate_color_palette(n_colors, seed=42):
    """Generate visually distinct colors for mapping."""
    random.seed(seed)
    colors = []
    
    # Use HSV color space for better distribution
    for i in range(n_colors):
        hue = (i * 137.5) % 360  # Golden angle for good distribution
        saturation = 65 + (i % 3) * 10  # Vary saturation slightly
        value = 75 + (i % 2) * 15  # Vary brightness slightly
        
        # Convert HSV to RGB
        import colorsys
        rgb = colorsys.hsv_to_rgb(hue/360, saturation/100, value/100)
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255)
        )
        colors.append(hex_color)
    
    return colors

# Create color assignments for each evolution category
if 'maintained_links' in locals():
    print("🎨 Generating Color Schemes for Maps")
    print("=" * 50)
    
    # For maintained sectors: simple unique color per sector
    n_maintained = len(maintained_links)
    maintained_colors = generate_color_palette(min(n_maintained, 100), seed=42)
    
    # For subdivided: group by 2010 sector, same color for all resulting 2022 sectors
    unique_subdivided_2010 = list(set(subdivided_links['SETOR_2010']))
    n_subdivided = len(unique_subdivided_2010)
    subdivided_colors_palette = generate_color_palette(min(n_subdivided, 100), seed=123)
    
    subdivided_color_map = {}
    for i, sector_2010 in enumerate(unique_subdivided_2010):
        color = subdivided_colors_palette[i % len(subdivided_colors_palette)]
        subdivided_color_map[sector_2010] = color
    
    # For aggregated: group by 2022 sector, same color for all source 2010 sectors
    unique_aggregated_2022 = list(set(aggregated_links['SETOR_2022']))
    n_aggregated = len(unique_aggregated_2022)
    aggregated_colors_palette = generate_color_palette(min(n_aggregated, 100), seed=456)
    
    aggregated_color_map = {}
    for i, sector_2022 in enumerate(unique_aggregated_2022):
        color = aggregated_colors_palette[i % len(aggregated_colors_palette)]
        aggregated_color_map[sector_2022] = color
    
    print(f"✅ Color schemes generated:")
    print(f"   • Maintained sectors: {len(maintained_colors)} colors")
    print(f"   • Subdivided groups: {len(subdivided_color_map)} colors")
    print(f"   • Aggregated groups: {len(aggregated_color_map)} colors")

else:
    print("⚠️ Cannot generate colors - evolution data not available")



### 3.9.1 Map Pair 1: Maintained Sectors (Unchanged 2010 → 2022)



In [ ]:
# Create side-by-side maps for MAINTAINED sectors
if 'maintained_links' in locals() and len(maintained_links) > 0:
    print("🗺️ Creating Maps: MAINTAINED SECTORS (Unchanged)")
    print("=" * 60)
    
    # Sample if too many sectors (for performance)
    max_sectors = 50000
    if len(maintained_links) > max_sectors:
        print(f"⚠️ Sampling {max_sectors} sectors from {len(maintained_links)} for visualization")
        maintained_sample = maintained_links.sample(n=max_sectors, random_state=42)
    else:
        maintained_sample = maintained_links
    
    # Get geometries for 2010 and 2022
    maintained_2010_sectors = set(maintained_sample['SETOR_2010'])
    maintained_2022_sectors = set(maintained_sample['SETOR_2022'])
    
    gdf_2010_maintained = gdf_2010[gdf_2010['sector_id'].isin(maintained_2010_sectors)].copy()
    gdf_2022_maintained = gdf_2022[gdf_2022['sector_id'].isin(maintained_2022_sectors)].copy()
    
    print(f"📊 Sectors to map:")
    print(f"   • 2010: {len(gdf_2010_maintained)} sectors")
    print(f"   • 2022: {len(gdf_2022_maintained)} sectors")
    
    # Assign colors - matching sectors get the same color
    sector_to_color = {}
    for i, (_, row) in enumerate(maintained_sample.iterrows()):
        color = maintained_colors[i % len(maintained_colors)]
        sector_to_color[row['SETOR_2010']] = color
        sector_to_color[row['SETOR_2022']] = color
    
    gdf_2010_maintained['color'] = gdf_2010_maintained['sector_id'].map(sector_to_color)
    gdf_2022_maintained['color'] = gdf_2022_maintained['sector_id'].map(sector_to_color)
    
    # Create side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # Plot 2010 map
    gdf_2010_maintained.plot(ax=ax1, color=gdf_2010_maintained['color'], 
                             edgecolor='black', linewidth=0.5, alpha=0.7)
    ax1.set_title('2010 Census - Maintained Sectors\n(Unchanged boundaries)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax1.axis('off')
    
    # Add text annotation
    ax1.text(0.02, 0.98, f'{len(gdf_2010_maintained)} sectors', 
             transform=ax1.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Plot 2022 map
    gdf_2022_maintained.plot(ax=ax2, color=gdf_2022_maintained['color'], 
                             edgecolor='black', linewidth=0.5, alpha=0.7)
    ax2.set_title('2022 Census - Maintained Sectors\n(Same boundaries as 2010)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax2.axis('off')
    
    # Add text annotation
    ax2.text(0.02, 0.98, f'{len(gdf_2022_maintained)} sectors', 
             transform=ax2.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Add overall title
    fig.suptitle('MAINTAINED SECTORS: Geographic Continuity 2010 ↔ 2022\n(Matching colors = Same sector)', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    # Save figure
    fig.savefig('../../results/historical_analysis/map_maintained_sectors.png', 
                dpi=300, bbox_inches='tight')
    
    print(f"\n✅ Map created!")
    print(f"   📌 Maintained sectors use matching colors between censuses")
    print(f"   📌 {len(maintained_sample)} sector pairs visualized")
    print(f"   💾 Saved: map_maintained_sectors.png")
    
else:
    print("⚠️ Cannot create maintained sectors maps - data not available")



### 3.9.2 Map Pair 2: Subdivided Sectors (1 sector in 2010 → Multiple in 2022)



In [ ]:
# Create side-by-side maps for SUBDIVIDED sectors
if 'subdivided_links' in locals() and len(subdivided_links) > 0:
    print("🗺️ Creating Maps: SUBDIVIDED SECTORS (1→N)")
    print("=" * 60)
    
    # Select representative subdivisions for visualization
    max_groups = 20000
    unique_subdivided_2010 = list(set(subdivided_links['SETOR_2010']))
    
    if len(unique_subdivided_2010) > max_groups:
        print(f"⚠️ Sampling {max_groups} subdivision groups from {len(unique_subdivided_2010)}")
        sampled_2010 = random.sample(unique_subdivided_2010, max_groups)
    else:
        sampled_2010 = unique_subdivided_2010
    
    subdivided_sample = subdivided_links[subdivided_links['SETOR_2010'].isin(sampled_2010)]
    
    # Get corresponding 2022 sectors
    subdivided_2010_sectors = set(subdivided_sample['SETOR_2010'])
    subdivided_2022_sectors = set(subdivided_sample['SETOR_2022'])
    
    gdf_2010_subdivided = gdf_2010[gdf_2010['sector_id'].isin(subdivided_2010_sectors)].copy()
    gdf_2022_subdivided = gdf_2022[gdf_2022['sector_id'].isin(subdivided_2022_sectors)].copy()
    
    print(f"📊 Sectors to map:")
    print(f"   • 2010: {len(gdf_2010_subdivided)} parent sectors")
    print(f"   • 2022: {len(gdf_2022_subdivided)} resulting sectors")
    
    # Assign colors - same color for parent and all children
    gdf_2010_subdivided['color'] = gdf_2010_subdivided['sector_id'].map(subdivided_color_map)
    
    # For 2022, find parent and use its color
    sector_2022_to_color = {}
    for _, row in subdivided_sample.iterrows():
        parent_color = subdivided_color_map.get(row['SETOR_2010'])
        if parent_color:
            sector_2022_to_color[row['SETOR_2022']] = parent_color
    
    gdf_2022_subdivided['color'] = gdf_2022_subdivided['sector_id'].map(sector_2022_to_color)
    
    # Create side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # Plot 2010 map
    gdf_2010_subdivided.plot(ax=ax1, color=gdf_2010_subdivided['color'], 
                             edgecolor='black', linewidth=1.2, alpha=0.7)
    ax1.set_title('2010 Census - Parent Sectors\n(Before subdivision)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax1.axis('off')
    
    # Add text annotation
    ax1.text(0.02, 0.98, f'{len(gdf_2010_subdivided)} parent sectors', 
             transform=ax1.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    # Plot 2022 map
    gdf_2022_subdivided.plot(ax=ax2, color=gdf_2022_subdivided['color'], 
                             edgecolor='white', linewidth=0.8, alpha=0.8)
    ax2.set_title('2022 Census - Subdivided Sectors\n(After subdivision)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax2.axis('off')
    
    # Add text annotation
    ax2.text(0.02, 0.98, f'{len(gdf_2022_subdivided)} child sectors', 
             transform=ax2.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    # Add overall title
    fig.suptitle('SUBDIVIDED SECTORS: Enhanced Spatial Resolution 2010 → 2022\n(Same color = All came from same 2010 sector)', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    # Save figure
    fig.savefig('../../results/historical_analysis/map_subdivided_sectors.png', 
                dpi=300, bbox_inches='tight')
    
    print(f"\n✅ Map created!")
    print(f"   📌 Each color represents one 2010 sector and its subdivisions")
    print(f"   📌 {len(gdf_2010_subdivided)} parent sectors → {len(gdf_2022_subdivided)} child sectors")
    print(f"   💾 Saved: map_subdivided_sectors.png")
    
else:
    print("⚠️ Cannot create subdivided sectors maps - data not available")



### 3.9.3 Map Pair 3: Aggregated Sectors (Multiple in 2010 → 1 sector in 2022)



In [ ]:
# Create side-by-side maps for AGGREGATED sectors
if 'aggregated_links' in locals() and len(aggregated_links) > 0:
    print("🗺️ Creating Maps: AGGREGATED SECTORS (N→1)")
    print("=" * 60)
    
    # Select representative aggregations for visualization
    max_groups = 200000
    unique_aggregated_2022 = list(set(aggregated_links['SETOR_2022']))
    
    if len(unique_aggregated_2022) > max_groups:
        print(f"⚠️ Sampling {max_groups} aggregation groups from {len(unique_aggregated_2022)}")
        sampled_2022 = random.sample(unique_aggregated_2022, max_groups)
    else:
        sampled_2022 = unique_aggregated_2022
    
    aggregated_sample = aggregated_links[aggregated_links['SETOR_2022'].isin(sampled_2022)]
    
    # Get corresponding sectors
    aggregated_2010_sectors = set(aggregated_sample['SETOR_2010'])
    aggregated_2022_sectors = set(aggregated_sample['SETOR_2022'])
    
    gdf_2010_aggregated = gdf_2010[gdf_2010['sector_id'].isin(aggregated_2010_sectors)].copy()
    gdf_2022_aggregated = gdf_2022[gdf_2022['sector_id'].isin(aggregated_2022_sectors)].copy()
    
    print(f"📊 Sectors to map:")
    print(f"   • 2010: {len(gdf_2010_aggregated)} source sectors")
    print(f"   • 2022: {len(gdf_2022_aggregated)} merged sectors")
    
    # Assign colors - same color for all sources and the resulting merged sector
    sector_2010_to_color = {}
    for _, row in aggregated_sample.iterrows():
        result_color = aggregated_color_map.get(row['SETOR_2022'])
        if result_color:
            sector_2010_to_color[row['SETOR_2010']] = result_color
    
    gdf_2010_aggregated['color'] = gdf_2010_aggregated['sector_id'].map(sector_2010_to_color)
    gdf_2022_aggregated['color'] = gdf_2022_aggregated['sector_id'].map(aggregated_color_map)
    
    # Create side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # Plot 2010 map
    gdf_2010_aggregated.plot(ax=ax1, color=gdf_2010_aggregated['color'], 
                             edgecolor='white', linewidth=0.8, alpha=0.8)
    ax1.set_title('2010 Census - Source Sectors\n(Before aggregation)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax1.axis('off')
    
    # Add text annotation
    ax1.text(0.02, 0.98, f'{len(gdf_2010_aggregated)} source sectors', 
             transform=ax1.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))
    
    # Plot 2022 map
    gdf_2022_aggregated.plot(ax=ax2, color=gdf_2022_aggregated['color'], 
                             edgecolor='black', linewidth=1.2, alpha=0.7)
    ax2.set_title('2022 Census - Aggregated Sectors\n(After merging)', 
                  fontsize=14, fontweight='bold', pad=20)
    ax2.axis('off')
    
    # Add text annotation
    ax2.text(0.02, 0.98, f'{len(gdf_2022_aggregated)} merged sectors', 
             transform=ax2.transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))
    
    # Add overall title
    fig.suptitle('AGGREGATED SECTORS: Boundary Consolidation 2010 → 2022\n(Same color = All merged into same 2022 sector)', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    # Save figure
    fig.savefig('../../results/historical_analysis/map_aggregated_sectors.png', 
                dpi=300, bbox_inches='tight')
    
    print(f"\n✅ Map created!")
    print(f"   📌 Each color represents one 2022 sector and its source 2010 sectors")
    print(f"   📌 {len(gdf_2010_aggregated)} source sectors → {len(gdf_2022_aggregated)} merged sectors")
    print(f"   💾 Saved: map_aggregated_sectors.png")
    
else:
    print("⚠️ Cannot create aggregated sectors maps - data not available")

